# superfly — FlyWire brain-graph toolkit demo

This notebook walks through `superfly`'s main features:
1. **Offline exploration** with mock synthetic data (no auth required)
2. **Real FlyWire queries** using your own CAVE token (optional)
3. **Disk caching** for fast repeated queries

## Setup: FlyWire/CAVE token (optional)

If you have a FlyWire/CAVE account, run the cell below **once per machine** to authenticate. Otherwise, skip to the mock data section — everything works offline without an account.

See [docs/SETUP.md](docs/SETUP.md) for full setup details.

In [ ]:
# Uncomment and run this cell ONCE to set up your CAVE token.
# Your browser will open for you to authenticate with Google.
# This only needs to be run once per machine.

# from caveclient import CAVEclient
# client = CAVEclient()
# client.auth.get_new_token()  # opens browser or prints URL
# # After authenticating, copy the token and paste it below:
# client.auth.save_token(token="<paste your token here>")
# print("Token saved to ~/.cloudvolume/secrets/cave-secret.json")

## Part 1: Offline exploration with MockDataSource

No auth required — explore the API using small synthetic fixture data.

In [ ]:
from superfly import FlyBrain
from superfly.datasources.mock import MockDataSource
import time

### Create a mock brain from synthetic data

In [ ]:
mock_ds = MockDataSource()

# List neurons in a region
al_r = mock_ds.get_annotations("AL_R", "region")
print(f"Neurons in AL_R: {len(al_r)}")
print(al_r[['root_id', 'cell_type', 'region']])

### Load a brain by region and build its connectivity graph

In [ ]:
brain = FlyBrain(datasource=mock_ds).load_region("AL_R", min_synapses=1)
print(brain)
print(f"Neurons: {len(brain.neuron_ids)}")
print(f"Edges (synaptic connections): {brain.adj.nnz}")

### Export to different formats

In [ ]:
# Dense numpy adjacency matrix
dense = brain.to_numpy()
print(f"Dense matrix shape: {dense.shape}")
print(f"Non-zero entries: {(dense > 0).sum()}")

In [ ]:
# NetworkX graph (requires networkx extra)
try:
    import networkx as nx
    g = brain.to_networkx()
    print(f"NetworkX graph:")
    print(f"  Nodes: {g.number_of_nodes()}")
    print(f"  Edges: {g.number_of_edges()}")
    print(f"  Density: {nx.density(g):.3f}")
except ImportError:
    print("networkx not installed. Install with: pip install superfly[viz]")

In [ ]:
# PyTorch Geometric graph (requires torch extra)
try:
    pyg = brain.to_pyg()
    print(f"PyTorch Geometric data:")
    print(f"  num_nodes: {pyg.num_nodes}")
    print(f"  edge_index: {pyg.edge_index.shape}")
    print(f"  edge_attr (synapse weights): {pyg.edge_attr.shape}")
except ImportError:
    print("torch/torch_geometric not installed. Install with: pip install superfly[torch]")

### Masking: constrain an arbitrary tensor to the fly's real wiring

`mask()` takes a tensor shaped like the brain's adjacency and zeroes out every entry where the fly has no corresponding synaptic connection — e.g. to constrain a randomly-initialized neural network's weights to only the connections that actually exist in the real connectome.

In [ ]:
try:
    import torch

    small = FlyBrain(datasource=mock_ds).load_cell_type("PN", min_synapses=1)
    print(f"Brain: {small}")

    n = len(small.neuron_ids)
    random_weights = torch.rand((n, n))
    masked_weights = small.mask(random_weights)

    print(f"Random weights, non-zero entries: {(random_weights != 0).sum().item()} / {n * n}")
    print(f"Masked weights, non-zero entries: {(masked_weights != 0).sum().item()} (constrained to real synapses)")
except ImportError:
    print("torch not installed. Install with: pip install superfly[torch]")

## Part 2: Real FlyWire queries

If you set up a CAVE token above, you can query the real FlyWire connectome.
Otherwise, the cells below will show you what would happen and skip the actual queries.

In [ ]:
from superfly.datasources.flybywire import FlyWireDataSource
from superfly.exceptions import AuthenticationError

# Try to authenticate. If it fails, we'll proceed with explanations only.
real_ds = FlyWireDataSource()
auth_available = False

try:
    real_ds.client  # This triggers the auth check
    auth_available = True
    print("✓ CAVE authentication successful!")
    print(f"  Datastack: {real_ds.dataset}")
    print(f"  Materialization: {real_ds.materialization}")
except AuthenticationError as e:
    print("⚠ CAVE authentication not available.")
    print("  Reason: " + str(e).split(".")[0])
    print("  Run the token setup cell above to authenticate.")
    print("  (This notebook will still work — we'll show examples below.)")

### Query by cell type (if authenticated)

In [ ]:
if auth_available:
    print("Querying for PN (Projection Neuron) cell type...")
    t0 = time.time()
    pn_brain = FlyBrain(datasource=real_ds).load_cell_type("PN", min_synapses=1)
    elapsed = time.time() - t0
    print(f"✓ Loaded in {elapsed:.1f}s: {pn_brain}")
else:
    print("Skipped: CAVE auth not available.")
    print("\nWhen authenticated, load_cell_type('PN', min_synapses=1) would return:")
    print("  A graph of ~700 projection neurons with their real connectome.")
    print("  First query is slow (can be 1-2+ minutes for hundreds of neurons --")
    print("  fetching adjacency is batched and network-bound), but the result")
    print("  is disk-cached. Repeated queries return in milliseconds from cache.")

### Query by region (if authenticated)

In [ ]:
if auth_available:
    print("Querying for AL_R (right antennal lobe) neuropil...")
    t0 = time.time()
    al_brain = FlyBrain(datasource=real_ds).load_region("AL_R", min_synapses=5)
    elapsed = time.time() - t0
    print(f"✓ Loaded in {elapsed:.1f}s: {al_brain}")
    
    # Show the neurobiological composition
    print(f"\nNeurobiological composition:")
    ann = real_ds.get_annotations("AL_R", "region")
    print(f"  Super classes: {dict(ann['super_class'].value_counts().head(3))}")
    print(f"  Top cell types: {dict(ann['cell_type'].value_counts().head(3))}")
else:
    print("Skipped: CAVE auth not available.")
    print("\nWhen authenticated, load_region('AL_R') would return:")
    print("  ~1,100 neurons in the antennal lobe, mostly sensory/olfactory.")
    print("  Region queries are approximate (see docs/SETUP.md for details).")
    print("  But they're biologically validated: AL_R is dominated by ORN")
    print("  (olfactory receptor neuron) and sensory cells, as expected.")

### Disk caching performance (if authenticated)

In [ ]:
if auth_available:
    from superfly.cache import CachedFlyWireDataSource
    cached_ds = CachedFlyWireDataSource()
    
    # First call: hits the network
    print("First call (hits network):")
    t0 = time.time()
    b1 = FlyBrain(datasource=cached_ds).load_cell_type("PN", min_synapses=5)
    t1 = time.time() - t0
    print(f"  {b1} in {t1:.1f}s")
    
    # Second call: cache hit, instant
    print("\nSecond call (cache hit):")
    t0 = time.time()
    b2 = FlyBrain(datasource=cached_ds).load_cell_type("PN", min_synapses=5)
    t2 = time.time() - t0
    print(f"  {b2} in {t2:.3f}s")
    print(f"\nSpeedup: {t1/t2:.0f}x faster on cache hit")
else:
    print("Skipped: CAVE auth not available.")
    print("\nWhen authenticated, repeated queries are disk-cached:")
    print("  First call: can be 1-2+ minutes for hundreds of neurons (network-bound)")
    print("  Repeat call: milliseconds (cache hit, often >10,000x faster)")
    print("  Cache location: ~/.cache/superfly/")

## Summary

- **Offline**: Use `MockDataSource` for development/testing with no auth required.
- **Real data**: Use `FlyWireDataSource` with a CAVE token for queries against the actual FlyWire connectome.
- **Caching**: Queries are automatically disk-cached for speed.
- **Exports**: Convert graphs to numpy arrays, networkx, or PyTorch Geometric formats.

See [docs/SETUP.md](docs/SETUP.md) for setup details, [README.md](README.md) for installation, and the source code in [src/superfly/](src/superfly/) for more advanced usage.